In [11]:
import pandas as pd
import json
from datetime import datetime

# ─── LOAD DATA ────────────────────────────────────────────────────────────────

def load_test_results(csv_file="test_results.csv"):
    df = pd.read_csv(csv_file)

    # ── Type conversions ──────────────────────────────────────────────────────
    df["current_glucose"]    = pd.to_numeric(df["current_glucose"],    errors="coerce")
    df["elapsed_seconds"]    = pd.to_numeric(df["elapsed_seconds"],    errors="coerce")
    df["attempts"]           = pd.to_numeric(df["attempts"],           errors="coerce")
    df["insulin_units_recommended"] = pd.to_numeric(df["insulin_units_recommended"], errors="coerce")
    df["is_safe"]            = df["is_safe"].astype(str).str.lower().map({"true": True, "false": False})
    df["run_timestamp"]      = pd.to_datetime(df["run_timestamp"], errors="coerce")

    print(f"✅ Loaded {len(df)} test results")
    print(f"   Columns: {list(df.columns)}\n")
    return df

df = load_test_results()



✅ Loaded 39 test results
   Columns: ['run_timestamp', 'test_id', 'label', 'current_glucose', 'glucose_category', 'meal_timing', 'oral_medication', 'insulin', 'long_acting_insulin', 'diet', 'elapsed_seconds', 'is_safe', 'attempts', 'medication_alert', 'insulin_units_recommended', 'insulin_timing_recommended', 'meal_recommended', 'exercise_recommended', 'violations', 'status', 'raw_output']



In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 1: OVERALL PASS / FAIL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

def analysis_overall_summary(df):
    print("=" * 70)
    print("ANALYSIS 1: OVERALL PASS / FAIL SUMMARY")
    print("=" * 70)

    total    = len(df)
    passed   = df["status"].eq("safe").sum()
    failed   = df["status"].eq("failed").sum()
    errored  = df["status"].eq("error").sum()
    pass_rate = round(passed / total * 100, 1) if total > 0 else 0

    print(f"  Total test runs : {total}")
    print(f"  ✅ Passed (safe): {passed}  ({pass_rate}%)")
    print(f"  ❌ Failed       : {failed}")
    print(f"  💥 Errored      : {errored}")
    print(f"  Avg elapsed     : {df['elapsed_seconds'].mean():.1f}s")
    print(f"  Max elapsed     : {df['elapsed_seconds'].max():.1f}s")
    print(f"  Min elapsed     : {df['elapsed_seconds'].min():.1f}s")
    print()

analysis_overall_summary(df)



ANALYSIS 1: OVERALL PASS / FAIL SUMMARY
  Total test runs : 39
  ✅ Passed (safe): 33  (84.6%)
  ❌ Failed       : 6
  💥 Errored      : 0
  Avg elapsed     : 65.2s
  Max elapsed     : 159.0s
  Min elapsed     : 25.3s



In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 2: PASS / FAIL BY GLUCOSE CATEGORY
# ══════════════════════════════════════════════════════════════════════════════

def analysis_by_glucose_category(df):
    print("=" * 70)
    print("ANALYSIS 2: PASS / FAIL BY GLUCOSE CATEGORY")
    print("=" * 70)

    summary = df.groupby("glucose_category").agg(
        total        = ("status", "count"),
        passed       = ("status", lambda x: (x == "safe").sum()),
        failed       = ("status", lambda x: (x == "failed").sum()),
        errored      = ("status", lambda x: (x == "error").sum()),
        avg_elapsed  = ("elapsed_seconds", "mean"),
        avg_attempts = ("attempts", "mean")
    ).reset_index()

    summary["pass_rate_%"] = (summary["passed"] / summary["total"] * 100).round(1)

    print(summary.to_string(index=False))
    print()

    # Flag any category below 80% pass rate
    low = summary[summary["pass_rate_%"] < 80]
    if not low.empty:
        print("⚠️  Categories below 80% pass rate:")
        for _, row in low.iterrows():
            print(f"   → {row['glucose_category']}: {row['pass_rate_%']}%")
    print()

analysis_by_glucose_category(df)



ANALYSIS 2: PASS / FAIL BY GLUCOSE CATEGORY
glucose_category  total  passed  failed  errored  avg_elapsed  avg_attempts  pass_rate_%
   hyperglycemia     13       9       4        0    76.473846      1.666667         69.2
    hypoglycemia      7       6       1        0    52.365714      1.500000         85.7
          normal     19      18       1        0    62.256316      1.388889         94.7

⚠️  Categories below 80% pass rate:
   → hyperglycemia: 69.2%



In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 3: PASS / FAIL BY MEAL TIMING
# ══════════════════════════════════════════════════════════════════════════════

def analysis_by_meal_timing(df):
    print("=" * 70)
    print("ANALYSIS 3: PASS / FAIL BY MEAL TIMING")
    print("=" * 70)

    summary = df.groupby("meal_timing").agg(
        total       = ("status", "count"),
        passed      = ("status", lambda x: (x == "safe").sum()),
        failed      = ("status", lambda x: (x == "failed").sum()),
        pass_rate   = ("status", lambda x: round((x == "safe").sum() / len(x) * 100, 1))
    ).reset_index().sort_values("pass_rate")

    print(summary.to_string(index=False))
    print()

analysis_by_meal_timing(df)



ANALYSIS 3: PASS / FAIL BY MEAL TIMING
                          meal_timing  total  passed  failed  pass_rate
                  AT long-acting time      2       1       1       50.0
                        BEFORE dinner      2       1       1       50.0
                   30min BEFORE lunch      3       2       1       66.7
               30min BEFORE breakfast      3       2       1       66.7
                         BEFORE lunch      3       2       1       66.7
                       AT dinner time      3       2       1       66.7
   1hr AFTER long-acting insulin time      2       2       0      100.0
                      1hr AFTER lunch      3       3       0      100.0
30min BEFORE long-acting insulin time      3       3       0      100.0
                  30min BEFORE dinner      3       3       0      100.0
                    2hrs AFTER dinner      1       1       0      100.0
                30min AFTER breakfast      1       1       0      100.0
          AT long-acting 

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 4: INSULIN RECOMMENDATION ACCURACY
# ══════════════════════════════════════════════════════════════════════════════

def analysis_insulin_accuracy(df):
    print("=" * 70)
    print("ANALYSIS 4: INSULIN RECOMMENDATION ACCURACY")
    print("=" * 70)

    # Expected units based on glucose dosing rules
    def expected_units(row):
        g = row["current_glucose"]
        insulin    = str(row.get("insulin", "")).lower()
        has_meal   = str(row.get("meal_timing", "")).upper()

        # If no short-acting insulin prescribed → 0
        if insulin not in ["yes", "y"]:
            return 0
        # If meal already taken (AFTER in label) → 0
        if "AFTER" in str(row.get("label", "")):
            return 0
        # Dosing rules
        if g < 70:   return 0   # hypoglycemia — no insulin
        if g <= 150: return 0   # in range — no correction
        if g <= 200: return 2
        if g <= 250: return 4
        return 6

    df["expected_units"]  = df.apply(expected_units, axis=1)
    df["units_match"]     = df["insulin_units_recommended"] == df["expected_units"]
    df["units_null"]      = df["insulin_units_recommended"].isna()

    total         = len(df)
    correct       = df["units_match"].sum()
    null_count    = df["units_null"].sum()
    accuracy      = round(correct / total * 100, 1)

    print(f"  Total tests          : {total}")
    print(f"  ✅ Correct units     : {correct}  ({accuracy}%)")
    print(f"  ❌ Incorrect units   : {total - correct - null_count}")
    print(f"  ⚠️  Null/missing units: {null_count}")
    print()

    # Show mismatches
    mismatches = df[~df["units_match"] & ~df["units_null"]][[
        "test_id", "label", "current_glucose",
        "expected_units", "insulin_units_recommended"
    ]]
    if not mismatches.empty:
        print("  Mismatched insulin recommendations:")
        print(mismatches.to_string(index=False))
    print()

    # Accuracy by glucose category
    cat_accuracy = df.groupby("glucose_category").agg(
        total   = ("units_match", "count"),
        correct = ("units_match", "sum"),
        nulls   = ("units_null", "sum")
    ).reset_index()
    cat_accuracy["accuracy_%"] = (cat_accuracy["correct"] / cat_accuracy["total"] * 100).round(1)
    print("  Insulin accuracy by glucose category:")
    print(cat_accuracy.to_string(index=False))
    print()

analysis_insulin_accuracy(df)

ANALYSIS 4: INSULIN RECOMMENDATION ACCURACY
  Total tests          : 39
  ✅ Correct units     : 34  (87.2%)
  ❌ Incorrect units   : 4
  ⚠️  Null/missing units: 1

  Mismatched insulin recommendations:
test_id                                                                                   label  current_glucose  expected_units  insulin_units_recommended
 TC-017                         HYPR-BRK-02 | Hyperglycemia | AT breakfast time | no medication              195               0                        2.0
 TC-020                          HYPR-LCH-03 | Hyperglycemia | 3hrs AFTER lunch | no medication              185               0                        2.0
 TC-022                            HYPR-DIN-02 | Hyperglycemia | AT dinner time | no medication              200               0                        2.0
 TC-024 EDGE-02 | Glucose exactly 150 (upper normal boundary) | BEFORE dinner | with medication              150               0                        2.0

  Insulin accuracy

In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 5: MEDICATION ALERT COVERAGE
# ══════════════════════════════════════════════════════════════════════════════

def analysis_medication_alert(df):
    print("=" * 70)
    print("ANALYSIS 5: MEDICATION ALERT COVERAGE")
    print("=" * 70)

    df["alert_fired"]    = df["medication_alert"].notna() & df["medication_alert"].ne("None") & df["medication_alert"].ne("")
    df["has_oral_med"]   = df["oral_medication"].str.lower().ne("none")
    df["meal_upcoming"]  = df["label"].str.contains("BEFORE", na=False)

    # Cases where alert SHOULD fire: oral medication + meal upcoming
    should_alert = df[df["has_oral_med"] & df["meal_upcoming"]]
    did_alert    = should_alert[should_alert["alert_fired"]]
    missed_alert = should_alert[~should_alert["alert_fired"]]

    coverage = round(len(did_alert) / len(should_alert) * 100, 1) if len(should_alert) > 0 else 0

    print(f"  Cases where alert should fire : {len(should_alert)}")
    print(f"  ✅ Alert fired correctly      : {len(did_alert)}  ({coverage}%)")
    print(f"  ❌ Alert missed               : {len(missed_alert)}")
    print()

    if not missed_alert.empty:
        print("  Tests where alert was missed:")
        print(missed_alert[["test_id", "label", "oral_medication", "meal_timing"]].to_string(index=False))
    print()

    # False positives: alert fired when meal already passed
    #meal_passed    = df[~df["meal_upcoming"] & df["alert_fired"]]
    #print(f"  ⚠️  Potential false alerts (meal already passed): {len(meal_passed)}")
    #if not meal_passed.empty:
    #    print(meal_passed[["test_id", "label", "medication_alert"]].to_string(index=False))
    #print()

analysis_medication_alert(df)

ANALYSIS 5: MEDICATION ALERT COVERAGE
  Cases where alert should fire : 16
  ✅ Alert fired correctly      : 16  (100.0%)
  ❌ Alert missed               : 0




In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 6: SAFETY AGENT VIOLATION ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def analysis_violations(df):
    print("=" * 70)
    print("ANALYSIS 6: SAFETY AGENT VIOLATION ANALYSIS")
    print("=" * 70)

    failed_df = df[df["status"] == "failed"].copy()

    if failed_df.empty:
        print("  ✅ No failures found.\n")
        return

    # Parse violations JSON strings
    def parse_violations(v):
        try:
            return json.loads(v) if pd.notna(v) else []
        except Exception:
            return [str(v)]

    failed_df["violations_list"] = failed_df["violations"].apply(parse_violations)

    # Flatten all violations into a single list
    all_violations = [
        v for vlist in failed_df["violations_list"] for v in vlist
    ]

    print(f"  Total failed tests : {len(failed_df)}")
    print(f"  Total violations   : {len(all_violations)}")
    print()

    # Categorize violations by keyword
    categories = {
        "Insulin null/missing":     ["null", "units", "insulin_recommendation"],
        "Carbohydrate violation":   ["carb", "carbohydrate"],
        "Exercise violation":       ["exercise"],
        "Hypoglycemia protocol":    ["hypoglycemia", "hypo", "70"],
        "Logical inconsistency":    ["inconsistency", "inconsistent", "logical"],
        "Parse error":              ["could not parse", "parse"],
    }

    cat_counts = {cat: 0 for cat in categories}
    uncategorized = []

    for v in all_violations:
        v_lower = v.lower()
        matched = False
        for cat, keywords in categories.items():
            if any(kw in v_lower for kw in keywords):
                cat_counts[cat] += 1
                matched = True
                break
        if not matched:
            uncategorized.append(v)

    print("  Violation breakdown by category:")
    for cat, count in sorted(cat_counts.items(), key=lambda x: -x[1]):
        bar = "█" * count
        print(f"    {cat:<35} {count:>3}  {bar}")
    if uncategorized:
        print(f"    {'Other':<35} {len(uncategorized):>3}")
    print()

    # Failures by glucose category
    fail_by_glucose = failed_df.groupby("glucose_category").size().reset_index(name="failures")
    print("  Failures by glucose category:")
    print(fail_by_glucose.to_string(index=False))
    print()

    # Failures by meal timing
    fail_by_timing = failed_df.groupby("meal_timing").size().reset_index(name="failures")
    print("  Failures by meal timing:")
    print(fail_by_timing.to_string(index=False))
    print()

analysis_violations(df)


ANALYSIS 6: SAFETY AGENT VIOLATION ANALYSIS
  Total failed tests : 6
  Total violations   : 12

  Violation breakdown by category:
    Insulin null/missing                  5  █████
    Carbohydrate violation                3  ███
    Exercise violation                    2  ██
    Hypoglycemia protocol                 0  
    Logical inconsistency                 0  
    Parse error                           0  
    Other                                 2

  Failures by glucose category:
glucose_category  failures
   hyperglycemia         4
    hypoglycemia         1
          normal         1

  Failures by meal timing:
           meal_timing  failures
30min BEFORE breakfast         1
    30min BEFORE lunch         1
        AT dinner time         1
   AT long-acting time         1
         BEFORE dinner         1
          BEFORE lunch         1



In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 7: RETRY LOOP EFFICIENCY
# ══════════════════════════════════════════════════════════════════════════════

def analysis_retry_efficiency(df):
    print("=" * 70)
    print("ANALYSIS 7: RETRY LOOP EFFICIENCY")
    print("=" * 70)

    attempt_df = df[df["attempts"].notna()].copy()

    if attempt_df.empty:
        print("  No attempt data available.\n")
        return

    one_shot  = (attempt_df["attempts"] == 1).sum()
    two_shot  = (attempt_df["attempts"] == 2).sum()
    three_plus = (attempt_df["attempts"] >= 3).sum()
    total     = len(attempt_df)

    print(f"  1 attempt (passed first try) : {one_shot}  ({round(one_shot/total*100,1)}%)")
    print(f"  2 attempts (1 retry)         : {two_shot}  ({round(two_shot/total*100,1)}%)")
    print(f"  3+ attempts                  : {three_plus}  ({round(three_plus/total*100,1)}%)")
    print(f"  Avg attempts per run         : {attempt_df['attempts'].mean():.2f}")
    print()

    # Retry rate by glucose category
    retry_by_cat = attempt_df.groupby("glucose_category").agg(
        avg_attempts = ("attempts", "mean"),
        max_attempts = ("attempts", "max"),
        pct_retried  = ("attempts", lambda x: round((x > 1).sum() / len(x) * 100, 1))
    ).reset_index()
    print("  Retry rate by glucose category:")
    print(retry_by_cat.to_string(index=False))
    print()

    # Tests that always needed retries
    always_retry = attempt_df[attempt_df["attempts"] > 1][[
        "test_id", "label", "glucose_category", "attempts", "status"
    ]]
    if not always_retry.empty:
        print("  Tests that required retries:")
        print(always_retry.to_string(index=False))
    print()

analysis_retry_efficiency(df)


ANALYSIS 7: RETRY LOOP EFFICIENCY
  1 attempt (passed first try) : 18  (54.5%)
  2 attempts (1 retry)         : 14  (42.4%)
  3+ attempts                  : 1  (3.0%)
  Avg attempts per run         : 1.48

  Retry rate by glucose category:
glucose_category  avg_attempts  max_attempts  pct_retried
   hyperglycemia      1.666667           2.0         66.7
    hypoglycemia      1.500000           2.0         50.0
          normal      1.388889           3.0         33.3

  Tests that required retries:
test_id                                                                                   label glucose_category  attempts status
 TC-002                  NORM-BRK-02 | Normal glucose | 30min AFTER breakfast | with medication           normal       2.0   safe
 TC-003                        NORM-BRK-03 | Normal glucose | AT breakfast time | no medication           normal       2.0   safe
 TC-008                           NORM-DIN-02 | Normal glucose | AT dinner time | no medication           

In [20]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 8: PERFORMANCE (LATENCY)
# ══════════════════════════════════════════════════════════════════════════════

def analysis_performance(df):
    print("=" * 70)
    print("ANALYSIS 8: PERFORMANCE (LATENCY)")
    print("=" * 70)

    print(f"  Overall:")
    print(f"    Mean   : {df['elapsed_seconds'].mean():.1f}s")
    print(f"    Median : {df['elapsed_seconds'].median():.1f}s")
    print(f"    P90    : {df['elapsed_seconds'].quantile(0.90):.1f}s")
    print(f"    Max    : {df['elapsed_seconds'].max():.1f}s")
    print(f"    Min    : {df['elapsed_seconds'].min():.1f}s")
    print()

    # Latency by glucose category
    lat_by_cat = df.groupby("glucose_category")["elapsed_seconds"].agg(
        ["mean", "median", "max", "min"]
    ).round(1).reset_index()
    lat_by_cat.columns = ["glucose_category", "mean_s", "median_s", "max_s", "min_s"]
    print("  Latency by glucose category:")
    print(lat_by_cat.to_string(index=False))
    print()

    # Latency by status
    lat_by_status = df.groupby("status")["elapsed_seconds"].agg(
        ["mean", "median", "max"]
    ).round(1).reset_index()
    lat_by_status.columns = ["status", "mean_s", "median_s", "max_s"]
    print("  Latency by status:")
    print(lat_by_status.to_string(index=False))
    print()

    # Slowest tests
    slowest = df.nlargest(5, "elapsed_seconds")[["test_id", "label", "elapsed_seconds", "status", "attempts"]]
    print("  Top 5 slowest tests:")
    print(slowest.to_string(index=False))
    print()

analysis_performance(df)

ANALYSIS 8: PERFORMANCE (LATENCY)
  Overall:
    Mean   : 65.2s
    Median : 55.0s
    P90    : 122.7s
    Max    : 159.0s
    Min    : 25.3s

  Latency by glucose category:
glucose_category  mean_s  median_s  max_s  min_s
   hyperglycemia    76.5      72.0  129.7   34.7
    hypoglycemia    52.4      56.2   81.6   28.6
          normal    62.3      43.1  159.0   25.3

  Latency by status:
status  mean_s  median_s  max_s
failed   111.6     116.7  154.5
  safe    56.8      43.3  159.0

  Top 5 slowest tests:
test_id                                                                                   label  elapsed_seconds status  attempts
 TC-007                    NORM-DIN-01 | Normal glucose | 30min BEFORE dinner | with medication           158.99   safe       1.0
 TC-004                     NORM-LCH-01 | Normal glucose | 30min BEFORE lunch | with medication           154.49 failed       NaN
 TC-024 EDGE-02 | Glucose exactly 150 (upper normal boundary) | BEFORE dinner | with medication   

In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 9: DIET PREFERENCE COMPLIANCE
# ══════════════════════════════════════════════════════════════════════════════

def analysis_diet_compliance(df):
    print("=" * 70)
    print("ANALYSIS 9: DIET PREFERENCE COMPLIANCE")
    print("=" * 70)

    # Forbidden foods per diet
    forbidden = {
        "Veg":   ["chicken", "turkey", "fish", "salmon", "tuna", "shrimp",
                  "beef", "pork", "lamb", "bacon", "sausage", "tilapia", "cod"],
        "Vegan": ["chicken", "turkey", "fish", "salmon", "tuna", "shrimp",
                  "beef", "pork", "eggs", "dairy", "yogurt", "paneer",
                  "cheese", "butter", "milk", "whey", "bacon", "sausage"]
    }

    def check_diet_violation(row):
        diet        = str(row.get("diet", "")).strip()
        meal_output = str(row.get("meal_recommended", "")).lower()
        if diet not in forbidden:
            return False
        return any(f in meal_output for f in forbidden[diet])

    df["diet_violation"] = df.apply(check_diet_violation, axis=1)

    by_diet = df.groupby("diet").agg(
        total          = ("diet_violation", "count"),
        violations     = ("diet_violation", "sum"),
        pass_rate      = ("status", lambda x: round((x == "safe").sum() / len(x) * 100, 1))
    ).reset_index()
    by_diet["compliance_%"] = (
        (by_diet["total"] - by_diet["violations"]) / by_diet["total"] * 100
    ).round(1)

    print(by_diet.to_string(index=False))
    print()

    # Show violations
    violations = df[df["diet_violation"]][[
        "test_id", "label", "diet", "meal_recommended"
    ]]
    if not violations.empty:
        print("  ⚠️  Diet compliance violations found:")
        print(violations.to_string(index=False))
    else:
        print("  ✅ No diet compliance violations detected.")
    print()

analysis_diet_compliance(df)

ANALYSIS 9: DIET PREFERENCE COMPLIANCE
   diet  total  violations  pass_rate  compliance_%
Non-Veg     37           0       86.5         100.0
    Veg      1           0      100.0         100.0
  Vegan      1           0        0.0         100.0

  ✅ No diet compliance violations detected.



In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 10: LONG-ACTING INSULIN ALERT COVERAGE
# ══════════════════════════════════════════════════════════════════════════════

def analysis_long_acting_insulin(df):
    print("=" * 70)
    print("ANALYSIS 10: LONG-ACTING INSULIN ALERT COVERAGE")
    print("=" * 70)

    lai_df = df[df["long_acting_insulin"].str.lower().ne("no") &
                df["long_acting_insulin"].notna()].copy()

    if lai_df.empty:
        print("  No long-acting insulin test cases found.\n")
        return

    # Check if medication alert mentions long-acting insulin
    lai_df["lai_alert_fired"] = lai_df["medication_alert"].str.lower().str.contains(
        "long.acting|long acting|basal|nightly|every night",
        regex=True, na=False
    )

    # LAI alert expected when current_time is near preferred time
    # (label contains "BEFORE long-acting" or "AT long-acting")
    lai_df["lai_due"] = lai_df["label"].str.contains(
        "BEFORE long-acting|AT long-acting", na=False
    )

    due      = lai_df[lai_df["lai_due"]]
    alerted  = due[due["lai_alert_fired"]]
    missed   = due[~due["lai_alert_fired"]]
    coverage = round(len(alerted) / len(due) * 100, 1) if len(due) > 0 else 0

    print(f"  LAI tests where alert due    : {len(due)}")
    print(f"  ✅ LAI alert fired correctly : {len(alerted)}  ({coverage}%)")
    print(f"  ❌ LAI alert missed          : {len(missed)}")
    print()

    if not missed.empty:
        print("  Missed LAI alerts:")
        print(missed[["test_id", "label", "long_acting_insulin", "medication_alert"]].to_string(index=False))
    print()

analysis_long_acting_insulin(df)

ANALYSIS 10: LONG-ACTING INSULIN ALERT COVERAGE
  LAI tests where alert due    : 7
  ✅ LAI alert fired correctly : 5  (71.4%)
  ❌ LAI alert missed          : 2

  Missed LAI alerts:
test_id                                                                            label     long_acting_insulin                                     medication_alert
 TC-030 LAI-02 | Hyperglycemia | 30min BEFORE long-acting insulin time | all medications Yes, every night 9PM ET Oral Medication: pre-meal | Insulin: yes | GLP-1: no
 TC-031  LAI-03 | Hypoglycemia | 30min BEFORE long-acting insulin time | all medications Yes, every night 9PM ET Oral Medication: pre-meal | Insulin: yes | GLP-1: no



In [23]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 11: HYPOGLYCEMIA SAFETY PROTOCOL COMPLIANCE
# ══════════════════════════════════════════════════════════════════════════════

def analysis_hypo_protocol(df):
    print("=" * 70)
    print("ANALYSIS 11: HYPOGLYCEMIA SAFETY PROTOCOL COMPLIANCE")
    print("=" * 70)

    hypo_df = df[df["glucose_category"] == "hypoglycemia"].copy()

    if hypo_df.empty:
        print("  No hypoglycemia test cases found.\n")
        return

    # Check: insulin units should be 0 for hypo
    hypo_df["insulin_zero"]    = hypo_df["insulin_units_recommended"] == 0
    hypo_df["meal_has_carbs"]  = hypo_df["meal_recommended"].str.lower().str.contains(
        "juice|glucose|carb|banana|raisin|honey|orange|apple", na=False
    )
    hypo_df["no_exercise"]     = ~hypo_df["exercise_recommended"].str.lower().str.contains(
        "recommend|suggest|engage|try", na=False
    )

    total = len(hypo_df)
    print(f"  Total hypoglycemia tests             : {total}")
    print(f"  ✅ Insulin correctly = 0             : {hypo_df['insulin_zero'].sum()} / {total}")
    print(f"  ✅ Fast-acting carbs recommended     : {hypo_df['meal_has_carbs'].sum()} / {total}")
    print(f"  ✅ Exercise correctly withheld       : {hypo_df['no_exercise'].sum()} / {total}")
    print()

    # Flag failures
    failures = hypo_df[
        ~hypo_df["insulin_zero"] |
        ~hypo_df["meal_has_carbs"] |
        ~hypo_df["no_exercise"]
    ][["test_id", "label", "insulin_units_recommended",
       "meal_recommended", "exercise_recommended"]]

    if not failures.empty:
        print("  ⚠️  Hypoglycemia protocol violations:")
        print(failures[["test_id", "label", "insulin_units_recommended"]].to_string(index=False))
    else:
        print("  ✅ All hypoglycemia protocols followed correctly.")
    print()

analysis_hypo_protocol(df)


ANALYSIS 11: HYPOGLYCEMIA SAFETY PROTOCOL COMPLIANCE
  Total hypoglycemia tests             : 7
  ✅ Insulin correctly = 0             : 7 / 7
  ✅ Fast-acting carbs recommended     : 7 / 7
  ✅ Exercise correctly withheld       : 3 / 7

  ⚠️  Hypoglycemia protocol violations:
test_id                                                                           label  insulin_units_recommended
 TC-010           HYPO-BRK-01 | Hypoglycemia | 30min BEFORE breakfast | with medication                        0.0
 TC-015                     HYPO-DIN-02 | Hypoglycemia | AT dinner time | no medication                        0.0
 TC-031 LAI-03 | Hypoglycemia | 30min BEFORE long-acting insulin time | all medications                        0.0
 TC-035    LAI-07 | Hypoglycemia | 1hr AFTER long-acting insulin time | all medications                        0.0



In [35]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 12: CONSOLIDATED SCORECARD
# ══════════════════════════════════════════════════════════════════════════════

def analysis_scorecard(df):
    print("=" * 70)
    print("ANALYSIS 12: CONSOLIDATED SCORECARD")
    print("=" * 70)

    total = len(df)

    # ── Metric calculations ───────────────────────────────────────────────────
    overall_pass     = round(df["status"].eq("safe").sum() / total * 100, 1)
    avg_latency      = round(df["elapsed_seconds"].mean(), 1)
    avg_attempts     = round(df["attempts"].mean(), 2)
    #null_insulin     = round(df["insulin_units_recommended"].isna().sum() / total * 100, 1)

    def expected_units(row):
        g = row["current_glucose"]
        if str(row.get("insulin", "")).lower() not in ["yes", "y"]: return 0
        if "AFTER" in str(row.get("label", "")): return 0
        if g < 70:   return 0
        if g <= 150: return 0
        if g <= 200: return 2
        if g <= 250: return 4
        return 6

    df["expected_units"] = df.apply(expected_units, axis=1)
    insulin_accuracy = round(
        (df["insulin_units_recommended"] == df["expected_units"]).sum() / total * 100, 1
    )

    df["alert_fired"] = (
        df["medication_alert"].notna() &
        df["medication_alert"].ne("None") &
        df["medication_alert"].ne("")
    )
    should_alert  = df[df["oral_medication"].str.lower().ne("none") & df["label"].str.contains("BEFORE")]
    alert_coverage = round(should_alert["alert_fired"].sum() / len(should_alert) * 100, 1) if len(should_alert) > 0 else 0

    hypo_df        = df[df["glucose_category"] == "hypoglycemia"]
    hypo_safe      = round(hypo_df["status"].eq("safe").sum() / len(hypo_df) * 100, 1) if len(hypo_df) > 0 else 0

    one_shot_rate  = round(df[df["attempts"].notna()]["attempts"].eq(1).sum() / df["attempts"].notna().sum() * 100, 1)

    forbidden = {
        "Veg":   ["chicken", "turkey", "fish", "salmon", "tuna", "shrimp", "beef", "pork"],
        "Vegan": ["chicken", "turkey", "fish", "eggs", "dairy", "yogurt", "paneer", "cheese"]
    }
    def check_diet_violation(row):
        diet = str(row.get("diet", "")).strip()
        meal = str(row.get("meal_recommended", "")).lower()
        if diet not in forbidden: return False
        return any(f in meal for f in forbidden[diet])
    diet_compliance = round((1 - df.apply(check_diet_violation, axis=1).sum() / total) * 100, 1)

    # ── Print scorecard ───────────────────────────────────────────────────────
    metrics = [
        ("Overall Pass Rate",           f"{overall_pass}%"    ),
        ("Insulin Accuracy",            f"{insulin_accuracy}%" ),
        #("Null Insulin Rate",           f"{null_insulin}%"    ),
        ("Medication Alert Coverage",   f"{alert_coverage}%"   ),
        ("Hypoglycemia Pass Rate",      f"{hypo_safe}%"        ),
        ("First-try Success Rate",      f"{one_shot_rate}%"    ),
        ("Diet Compliance",             f"{diet_compliance}%"  ),
        ("Avg Latency",                 f"{avg_latency}s"      ),
        ("Avg Attempts per Run",        f"{avg_attempts}"     ),
    ]
    #print(metrics)
    print(f"  {'Metric':<35} {'Value':>10} ")
    #print(f"  {'-'*60}")
    for name, value in metrics:
      print(f"  {name:<35} {value:>10}" )


analysis_scorecard(df)

ANALYSIS 12: CONSOLIDATED SCORECARD
  Metric                                   Value 
  Overall Pass Rate                        84.6%
  Insulin Accuracy                         87.2%
  Medication Alert Coverage               100.0%
  Hypoglycemia Pass Rate                   85.7%
  First-try Success Rate                   54.5%
  Diet Compliance                         100.0%
  Avg Latency                              65.2s
  Avg Attempts per Run                      1.48
